# Phase-Transition-Hardness: Introduction

This notebook provides an introduction to the Phase-Transition-Hardness codebase and demonstrates basic usage.

## Overview

The Phase-Transition-Hardness project formalizes the correspondence between computational hardness in random constraint satisfaction problems (CSPs) and phase-transition phenomena in statistical mechanics.

### Key Concepts

1. **Random k-SAT**: A canonical NP-complete problem where we seek assignments to n Boolean variables that satisfy m clauses
2. **Phase Transitions**: Sharp transitions in solution space structure as constraint density α = m/n varies
3. **Computational Hardness**: Exponential growth of solver runtime near critical thresholds
4. **Barrier-Hardness Correspondence**: Conjecture that hardness scales with free-energy barriers in the solution landscape

### Critical Thresholds (for 3-SAT)

| Threshold | Symbol | Value | Significance |
|-----------|--------|-------|--------------|
| Clustering | α_d | ≈ 3.86 | Solution space shatters into clusters |
| Rigidity | α_r | ≈ 4.00 | Variables become frozen within clusters |
| Condensation | α_c | ≈ 4.10 | Gibbs measure concentrates on few clusters |
| Satisfiability | α_s | ≈ 4.267 | Transition from SAT to UNSAT |
| Hardness Peak | α* | ≈ 4.20 | Maximum computational difficulty |

## Setup

First, let's import the necessary modules and configure the environment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

# Import our modules
from src.instance_generator import generate_ksat_instance, generate_instance_batch
from src.energy_model import (
    ALPHA_D, ALPHA_R, ALPHA_C, ALPHA_S,
    rs_entropy_density,
    cluster_complexity,
    barrier_density,
    frozen_fraction
)
from src.hardness_metrics import dpll_solve, measure_hardness
from src.phase_transition import estimate_psat_single

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Generating Random k-SAT Instances

Let's start by generating a simple random 3-SAT instance.

In [ ]:
# Generate a small instance
n = 20  # Number of variables
alpha = 3.0  # Constraint density
k = 3  # Clause length (3-SAT)

instance = generate_ksat_instance(n=n, alpha=alpha, k=k, seed=42)

print(f"Generated {k}-SAT instance:")
print(f"  Variables (n): {instance['n']}")
print(f"  Clauses (m): {instance['m']}")
print(f"  Constraint density (α): {instance['alpha']}")
print(f"\nFirst 5 clauses:")
for i, clause in enumerate(instance['clauses'][:5]):
    print(f"  Clause {i+1}: {clause}")

## 2. Solving with DPLL

Now let's solve this instance using the DPLL algorithm.

In [ ]:
# Solve the instance
result = dpll_solve(instance, max_decisions=10000)

print(f"DPLL Result:")
print(f"  Satisfiable: {result['satisfiable']}")
print(f"  Decisions: {result['decisions']}")

if result['satisfiable']:
    print(f"\nSatisfying assignment (first 10 variables):")
    for var in range(1, min(11, n+1)):
        print(f"  x_{var} = {result['assignment'][var]}")

## 3. Measuring Hardness

The hardness metric γ = log(T) / n measures the exponential growth rate of solver runtime.

In [ ]:
# Measure hardness
hardness = measure_hardness(instance, solver='dpll', max_decisions=10000)
print(f"Hardness density γ = {hardness:.4f}")
print(f"Interpretation: Runtime grows as T ~ exp({hardness:.4f} × n)")

## 4. Theoretical Order Parameters

Let's visualize the theoretical predictions from statistical mechanics.

In [ ]:
# Generate alpha values
alphas = np.linspace(2.5, 5.5, 200)

# Compute order parameters
entropy = [rs_entropy_density(a) for a in alphas]
complexity = [cluster_complexity(a) for a in alphas]
barriers = [barrier_density(a) for a in alphas]
frozen = [frozen_fraction(a) for a in alphas]

# Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Entropy
axes[0, 0].plot(alphas, entropy, 'b-', linewidth=2)
axes[0, 0].axvline(ALPHA_S, color='r', linestyle='--', label=f'α_s = {ALPHA_S}')
axes[0, 0].set_xlabel('Constraint Density α')
axes[0, 0].set_ylabel('Entropy Density s(α)')
axes[0, 0].set_title('Replica Symmetric Entropy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Complexity
axes[0, 1].plot(alphas, complexity, 'g-', linewidth=2)
axes[0, 1].axvline(ALPHA_D, color='r', linestyle='--', label=f'α_d = {ALPHA_D}')
axes[0, 1].axvline(ALPHA_C, color='orange', linestyle='--', label=f'α_c = {ALPHA_C}')
axes[0, 1].set_xlabel('Constraint Density α')
axes[0, 1].set_ylabel('Cluster Complexity Σ(α)')
axes[0, 1].set_title('Configurational Entropy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Barrier density
axes[1, 0].plot(alphas, barriers, 'm-', linewidth=2)
axes[1, 0].axvline(ALPHA_D, color='r', linestyle='--', label=f'α_d = {ALPHA_D}')
axes[1, 0].axvline(4.2, color='purple', linestyle=':', label='α* ≈ 4.2')
axes[1, 0].axvline(ALPHA_S, color='orange', linestyle='--', label=f'α_s = {ALPHA_S}')
axes[1, 0].set_xlabel('Constraint Density α')
axes[1, 0].set_ylabel('Barrier Density b(α)')
axes[1, 0].set_title('Free-Energy Barrier Density')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Frozen fraction
axes[1, 1].plot(alphas, frozen, 'c-', linewidth=2)
axes[1, 1].axvline(ALPHA_D, color='r', linestyle='--', label=f'α_d = {ALPHA_D}')
axes[1, 1].axvline(ALPHA_R, color='g', linestyle='--', label=f'α_r = {ALPHA_R}')
axes[1, 1].axvline(ALPHA_S, color='orange', linestyle='--', label=f'α_s = {ALPHA_S}')
axes[1, 1].set_xlabel('Constraint Density α')
axes[1, 1].set_ylabel('Frozen Fraction')
axes[1, 1].set_title('Fraction of Frozen Variables')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('order_parameters.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. P_sat Estimation

Let's estimate the probability that a random instance is satisfiable at different constraint densities.

In [ ]:
# Estimate P_sat for different alphas
alphas_test = np.linspace(3.0, 6.0, 10)
psat_estimates = []

print("Estimating P_sat (this may take a minute)...")
for alpha in alphas_test:
    psat = estimate_psat_single(
        n=30,
        alpha=alpha,
        n_instances=50,
        k=3,
        master_seed=42,
        solver='dpll'
    )
    psat_estimates.append(psat)
    print(f"  α = {alpha:.2f}: P_sat ≈ {psat:.3f}")

# Plot
plt.figure(figsize=(10, 6))
plt.plot(alphas_test, psat_estimates, 'bo-', linewidth=2, markersize=8)
plt.axvline(ALPHA_S, color='r', linestyle='--', label=f'α_s = {ALPHA_S} (theory)')
plt.xlabel('Constraint Density α', fontsize=12)
plt.ylabel('P_sat (n=30)', fontsize=12)
plt.title('Satisfiability Probability vs Constraint Density', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.ylim(-0.05, 1.05)
plt.savefig('psat_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Next Steps

Now that you understand the basics, you can:

1. **Run full experiments**: See `02_alpha_sweep.ipynb` for complete alpha sweep
2. **Analyze scaling**: See `03_finite_size_scaling.ipynb` for FSS analysis
3. **Validate results**: Run `src/validation.py` to check against manuscript predictions
4. **Explore the code**: Browse the `src/` directory for implementation details

### Key Takeaways

- Random k-SAT exhibits sharp phase transitions in solution space structure
- Computational hardness peaks in the "shattered phase" between α_d and α_s
- Free-energy barriers govern the exponential scaling of solver runtime
- All experiments are fully reproducible from a single master seed